# PubMed Biomedical Metadata — Exploratory Analysis

A tour of the dataset: scale, coverage over time, journals, MeSH topics, author teams, abstracts, and how field completeness changes across the decades.

Run on the local clean corpus, abstract text is present and is analysed directly (length, structure, vocabulary). The same notebook also runs on the published Kaggle dataset, which is metadata-only — there the abstract-text sections detect the absence and skip gracefully, while every metadata section still works. A single flag set in the setup cell, `HAS_ABSTRACTS`, controls this.

## Setup

In [ ]:
import os, glob, re, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING — pick ONE option
# =====================================================================

# ---- OPTION A: LOCAL (active) -----------------------------------------
# Notebook runs from notebooks/, data sits one level up in the project root.
# Loads the full clean corpus (with abstracts); 2026 live-edge records are trimmed in 2b.
ROOT = os.path.dirname(os.getcwd())                       # go up one folder
DATA_DIR = os.path.join(ROOT, "data", "2_clean")          # full clean corpus (with abstracts)

# ---- OPTION B: KAGGLE (commented out — uncomment when running on Kaggle) ----
# # The published Kaggle data is metadata-only and already filtered to <= 2025, so 2b is a no-op.
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input — attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])

# =====================================================================
df = pd.read_parquet(DATA_DIR)

# does this copy actually contain abstract text? (True locally, False on the metadata-only export)
HAS_ABSTRACTS = bool((df['abstract'].fillna('').str.len() > 0).any())

print(f"loaded {len(df):,} records from {DATA_DIR}")
print("abstract text present:", HAS_ABSTRACTS)
print("columns:", list(df.columns))
df.head(3)

## 1. Scale and schema
One row per article, keyed by PubMed ID (`uid`), with bibliographic metadata and flattened list fields (authors, affiliations, MeSH descriptors, keywords).

In [ ]:
print(f"records:       {len(df):,}")
print(f"unique PMIDs:  {df['uid'].nunique():,}")
print(f"year range:    {int(df['year'].min())}–{int(df['year'].max())}")
print(f"columns:       {df.shape[1]}")
df.dtypes

## 2. Publications per year

### 2a. Volume trend
Annual volume grows steadily from the mid-1990s, accelerating through the 2010s, with a conspicuous dip around 2012–2015 (highlighted) and a small live-edge tail at the most recent year.

In [ ]:
per_year = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.lineplot(x=per_year.index, y=per_year.values, marker="o", color="#1d6fb8")
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.15)
plt.axvline(2012, color="#e07a5f", ls="--", lw=1.2)
plt.axvline(2015, color="#e07a5f", ls="--", lw=1.2)
seg = per_year.loc[2012:2015]
sns.lineplot(x=seg.index, y=seg.values, marker="o", color="#e07a5f", lw=2.5)
plt.title("Articles per year (2012–2015 dip highlighted)")
plt.xlabel("year"); plt.ylabel("articles")
plt.tight_layout(); plt.show()

### 2b. Trim the live edge (drop 2026)
The chart shows a tiny tail in 2026: a handful of electronic / ahead-of-print records plus still-accruing months — the unstable "live edge" of PubMed. They are dropped for a clean, frozen 1994–2025 view. This affects only the analysis; the underlying clean corpus is unchanged. (On Kaggle the data is already ≤ 2025, so this is a no-op.)

In [ ]:
MAX_YEAR = 2025
before = len(df)
df = df[df['year'] <= MAX_YEAR].copy()
print(f"trimmed to year <= {MAX_YEAR}: {len(df):,} records "
      f"(removed {before - len(df):,} live-edge rows)")
print(f"year range now: {int(df['year'].min())}–{int(df['year'].max())}")

## 2c. Diagnosing the 2012–2015 dip
Counts fall from a 2012 peak to a 2014 low (−32.5%), then recover — against otherwise steady 2–6%/year growth. The cells below establish the magnitude and isolate the cause.

### 2c-i. Magnitude (year-over-year change)

In [ ]:
per_year = df['year'].value_counts().sort_index()
stats = pd.DataFrame({"articles": per_year})
stats["yoy_change"] = stats["articles"].diff().astype("Int64")
stats["yoy_pct"] = (stats["articles"].pct_change() * 100).round(1)
print(stats.to_string())

### 2c-ii. By date precision
If the dip hit one date-format more than others it would point to a dating change. It does not — all three precision types fall together.

In [ ]:
prec_counts = df.groupby(['year', 'pubdate_precision']).size().unstack(fill_value=0)
plt.figure(figsize=(12, 4))
for col in prec_counts.columns:
    sns.lineplot(x=prec_counts.index, y=prec_counts[col], marker="o", label=col)
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.10)
plt.title("Article counts by date-precision, per year")
plt.xlabel("year"); plt.ylabel("articles"); plt.legend(title="precision")
plt.tight_layout(); plt.show()
print(prec_counts.loc[2011:2016].to_string())

### 2c-iii. By journal (the decisive test)
If the dip is uniform across journals it is systemic; if concentrated, specific sources drove it. Established US journals nearly vanish from the 2014 bucket while new megajournals grow — a wildly non-uniform ratio.

In [ ]:
top_j = df['journal'].value_counts().head(15).index
piv = df[df['journal'].isin(top_j)].groupby(['year', 'journal']).size().unstack(fill_value=0)
ratio = (piv.loc[2014] / piv.loc[2012]).sort_values()
print("2014/2012 article ratio for top-15 journals:")
print(ratio.round(2).to_string())
print(f"\nmedian ratio: {ratio.median():.2f}  "
      f"(uniform ~same = systemic; wide spread = specific journals drove it)")

### 2c-iv. Affiliation coverage (circular — shown for completeness)
This cannot detect the dip: the query *requires* a US affiliation, so coverage is ~100% every year by construction. Included only to make that explicit.

In [ ]:
cov = df.assign(has_aff=df['affiliations'].map(lambda v: len(v) > 0)).groupby('year')['has_aff'].mean() * 100
print(cov.round(3).to_string())
print("\nNote: ~100% every year is expected — the query requires an affiliation, so this cannot detect the dip.")

**Conclusion — the dip is an affiliation-filter artifact, not a real decline.** All precision types drop ~35% together in 2014 (not a date-format issue). By journal the cause is clear: established US journals nearly vanish from the 2014 bucket (Journal of Biological Chemistry, Blood, Cancer ≈ 0.00× their 2012 count; PNAS 0.01×) while newly launched megajournals grow (Scientific Reports 4×, Nature Communications 6.5×). Those journals plainly did not stop publishing US research in 2014 — the `USA[Affiliation]` filter failed to match their 2013–2014 records, coinciding with PubMed's affiliation-indexing transition (~2013–2014, first-author-only → all-author affiliations). The harvest matches PubMed's own per-month counts exactly, so the dip is a faithful property of the filtered query — a metadata artifact, **not** a real change in output. Volume in 2013–2015 should be treated as a lower bound, not a trend.

## 3. Dates

### 3a. Date precision
Most records carry imprecise publication dates (year-only or year+month). Use `pubdate_precision` and restrict to `full_date` for month-level work.

In [ ]:
prec = df['pubdate_precision'].value_counts()
plt.figure(figsize=(7, 4))
sns.barplot(x=prec.index, y=prec.values, color="#1d6fb8")
plt.title("Publication-date precision"); plt.ylabel("records"); plt.xlabel("")
plt.tight_layout(); plt.show()
print((prec / len(df) * 100).round(1).astype(str) + " %")

### 3b. Indexing lag (publication date vs harvest month)
`source_month` is the month-bucket each record was harvested under; `pubdate` is its publication date. Their gap is a proxy for indexing lag, informative about the recent-year tail where records are still being indexed. Uses only `full_date` records (a precise pubdate is needed).

In [ ]:
fd = df[df['pubdate_precision'] == 'full_date']
pm = pd.to_datetime(fd['pubdate'], errors='coerce').dt.to_period('M')
sm = pd.PeriodIndex(fd['source_month'], freq='M')
lag = (sm.astype('int64') - pm.astype('int64'))
lag = lag[(lag >= -2) & (lag <= 60)]            # drop nonsensical extremes
print(f"indexing lag (months) — median {int(lag.median())}, mean {lag.mean():.1f}")
plt.figure(figsize=(10, 3))
sns.histplot(lag, bins=40, color="#8a5fb0")
plt.title("Harvest-month minus publication-month (indexing lag, months)")
plt.xlabel("months"); plt.tight_layout(); plt.show()

## 4. Journals

### 4a. Top journals
The most prolific titles in the corpus.

In [ ]:
print("distinct journals:", df['journal'].nunique())
top = df['journal'].value_counts().head(15)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
plt.title("Top 15 journals by article count"); plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 4b. Concentration and new journals
How much of the corpus the top-N journals capture, and how many journals first appear each year. "First appearance" is within this corpus, not a journal's true founding year.

In [ ]:
total = len(df)
top_counts = df['journal'].value_counts()
print("journal concentration:")
for n in [5, 10, 20, 50, 100]:
    print(f"  top {n:>3} journals = {top_counts.head(n).sum()/total*100:5.1f}% of all articles")

first_seen = df.groupby('journal')['year'].min()
new_per_year = first_seen.value_counts().sort_index()
plt.figure(figsize=(12, 4))
sns.lineplot(x=new_per_year.index, y=new_per_year.values, marker="o", color="#2a9d5c")
plt.title("Journals by first appearance in the corpus (proxy, not founding year)")
plt.xlabel("year"); plt.ylabel("journals first seen"); plt.tight_layout(); plt.show()
print(new_per_year.tail(10).to_string())

### 4c. Journal turnover — rank over time (bump chart)
How the leading journals rise and fall in rank. Established society journals lose ground while large open-access megajournals climb — the same turnover that drives the 2014 affiliation artifact in §2c.

Note: pre-2014 journal shares are affected by that same artifact — established journals are undercounted after ~2012 because their records stopped matching the US-affiliation filter, not because they stopped publishing. Read the turnover as "what the filtered corpus contains," not the real journal landscape.

In [ ]:
TOPN = 10
overall_top = df['journal'].value_counts().head(TOPN).index
counts = (df[df['journal'].isin(overall_top)]
          .groupby(['year', 'journal']).size().unstack(fill_value=0))
ranks = counts.rank(axis=1, ascending=False, method='min')   # 1 = most articles that year

plt.figure(figsize=(13, 6))
for j in overall_top:
    plt.plot(ranks.index, ranks[j], marker='o', markersize=4, linewidth=1.8, label=j)
plt.gca().invert_yaxis()
plt.yticks(range(1, TOPN + 1))
plt.title("Top journals — yearly rank (bump chart; 1 = most articles)")
plt.xlabel("year"); plt.ylabel("rank")
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

### 4d. Journal share over time
The same turnover as a share of annual output, so the magnitude of each journal's rise or fall is visible (rank alone hides how big the gaps are).

In [ ]:
share = counts.div(df.groupby('year').size(), axis=0) * 100
share.plot.area(figsize=(13, 5), alpha=0.8)
plt.title("Top-10 journals — share of annual articles (%)")
plt.xlabel("year"); plt.ylabel("% of that year's articles")
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
plt.tight_layout(); plt.show()
print("share %, selected years:")
print(share.loc[share.index.isin([1995, 2005, 2015, 2025])].round(2).to_string())

### 4e. Journal concentration — Lorenz curve and Gini
How unequally articles are distributed across all journals. The Lorenz curve plots cumulative share of articles against cumulative share of journals; Gini summarises it (0 = perfectly even, 1 = all articles in one journal).

In [ ]:
trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz   # NumPy 2.x renamed trapz
jc = df['journal'].value_counts().sort_values().values
cum_articles = np.cumsum(jc) / jc.sum()
cum_journals = np.arange(1, len(jc) + 1) / len(jc)
gini = 1 - 2 * trapz(cum_articles, cum_journals)
half = int((cum_articles >= 0.5).argmax()); pct_journals = (len(jc) - half) / len(jc) * 100

plt.figure(figsize=(6, 6))
plt.plot(cum_journals, cum_articles, color="#1d6fb8", lw=2)
plt.plot([0, 1], [0, 1], ls="--", color="grey")
plt.title(f"Journal concentration (Gini = {gini:.2f})")
plt.xlabel("cumulative share of journals"); plt.ylabel("cumulative share of articles")
plt.tight_layout(); plt.show()
print(f"Gini = {gini:.2f}; the top {pct_journals:.0f}% of journals account for 50% of all articles")

## 5. Authors

### 5a. Authors per paper
Right-skewed: most papers have a handful of authors, with a long tail of large collaborations.

In [ ]:
print(df['n_authors'].describe().astype(int))
plt.figure(figsize=(10, 4))
sns.histplot(df['n_authors'].clip(upper=30), bins=30, color="#1d6fb8")
plt.title("Authors per paper (clipped at 30)"); plt.xlabel("authors"); plt.ylabel("papers")
plt.tight_layout(); plt.show()

### 5b. Team size over time
The composition shifts toward larger teams — solo work shrinks, large collaborations grow.

In [ ]:
band = pd.cut(df['n_authors'], [0, 1, 5, 20, 10**9], labels=["solo", "2–5", "6–20", "21+"])
collab = df.assign(band=band).groupby(['year', 'band'], observed=True).size().unstack(fill_value=0)
collab_pct = collab.div(collab.sum(axis=1), axis=0) * 100

collab_pct.plot.area(figsize=(12, 4), color=["#d9534f", "#1d6fb8", "#2a9d5c", "#8a5fb0"])
plt.title("Team-size composition by year (%)"); plt.xlabel("year"); plt.ylabel("% of articles")
plt.legend(title="authors", loc="lower left"); plt.tight_layout(); plt.show()
print("share by band, selected years:")
print(collab_pct.loc[collab_pct.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

### 5c. Correlation among numeric features
How the numeric fields relate. Correlations are expected to be weak; these are descriptive, not inferential (no multiple-comparison correction). Any correlation involving `year` is confounded by the indexing artifacts above (the 2014 dip, the MeSH-depth shift, the keyword/COI onset) — read `year` rows with caution.

In [ ]:
num_cols = ['n_authors', 'n_mesh', 'n_keywords', 'abstract_len', 'year']
corr = df[num_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation among numeric features (descriptive)'); plt.tight_layout(); plt.show()

## 6. MeSH topics

### 6a. Top descriptors
Most frequent MeSH descriptors — dominated by the human-subject scope of the corpus.

In [ ]:
mc = collections.Counter(d for lst in df['mesh_descriptors'] for d in lst)
print("distinct MeSH descriptors:", len(mc))
top = pd.Series(dict(mc.most_common(15)))[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#2a9d5c")
plt.title("Top 15 MeSH descriptors"); plt.xlabel("occurrences"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 6b. Top descriptors over time (raw counts)
Raw counts of the leading descriptors per year. These largely track total volume — see 6e for the volume-normalized version.

In [ ]:
TOP_N = 15
top_descriptors = [d for d, _ in mc.most_common(TOP_N)]
mesh_exploded = (df[["year", "mesh_descriptors"]]
                 .explode("mesh_descriptors")
                 .rename(columns={"mesh_descriptors": "descriptor"}))
mesh_exploded = mesh_exploded[mesh_exploded["descriptor"].isin(top_descriptors)]
mesh_ts = mesh_exploded.groupby(["year", "descriptor"]).size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(13, 6))
for desc in top_descriptors:
    sub = mesh_ts[mesh_ts["descriptor"] == desc]
    ax.plot(sub["year"], sub["count"], label=desc, linewidth=1.5)
ax.set_title("Top 15 MeSH descriptors — raw article count per year")
ax.set_xlabel("year"); ax.set_ylabel("articles")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

### 6c. Descriptors per article over time
The structural shift: descriptor depth is stable until ~2019, then drops.

In [ ]:
mesh_year = df.groupby('year')['n_mesh'].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=mesh_year.index, y=mesh_year.values, marker="o", color="#2a9d5c")
plt.title("Mean MeSH descriptors per article, by year"); plt.xlabel("year"); plt.ylabel("mean # MeSH")
plt.tight_layout(); plt.show()
print(mesh_year.round(2).to_string())

### 6d. Volume vs MeSH depth side by side
Total articles and mean descriptors per article, to confirm the depth decline is independent of volume.

In [ ]:
mesh_coverage = df.groupby("year").agg(
    total=("uid", "count"),
    with_mesh=("n_mesh", lambda x: (x > 0).sum()),
    avg_mesh=("n_mesh", "mean")
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mesh_coverage["year"], mesh_coverage["total"], color="#1d6fb8")
axes[0].set_title("total articles per year"); axes[0].set_xlabel("year")
axes[1].plot(mesh_coverage["year"], mesh_coverage["avg_mesh"], color="#e0853f")
axes[1].set_title("avg MeSH descriptors per article"); axes[1].set_xlabel("year")
plt.tight_layout(); plt.show()

### 6e. Top descriptors normalized per 1,000 articles
Dividing by articles per year shows which topics genuinely rose or fell in *prevalence*, independent of corpus growth.

In [ ]:
art_per_year = df.groupby('year').size()
mex = df[['year', 'mesh_descriptors']].explode('mesh_descriptors')
top_norm = mex['mesh_descriptors'].value_counts().head(10).index
norm = (mex[mex['mesh_descriptors'].isin(top_norm)]
        .groupby(['year', 'mesh_descriptors']).size().unstack(fill_value=0)
        .div(art_per_year, axis=0) * 1000)

plt.figure(figsize=(13, 6))
for d in top_norm:
    plt.plot(norm.index, norm[d], label=d, lw=1.5)
plt.title("Top MeSH descriptors — prevalence per 1,000 articles per year")
plt.xlabel("year"); plt.ylabel("per 1,000 articles")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

MeSH coverage is 100% across all years — every record carries at least one descriptor (PubMed releases fully-indexed records). But the **mean number of descriptors per article is stable at ~13 through 2019, then falls to ~8 by 2022–2023**, with a partial rebound in 2024–2025. This matches NLM's transition to automated indexing (~2021–2022, fewer descriptors) plus indexing lag on recent years — not a content change. MeSH-breadth analyses should segment by era and treat post-2019 depth cautiously.

## 7. Field completeness over time

### 7a. Coverage trend (with auto-detected thresholds)
Keywords (from ~2012) and conflict-of-interest statements (from ~2017) became standard only gradually. The "valid from" thresholds are computed from the data rather than hardcoded, so they stay correct if the corpus updates.

In [ ]:
by_year = df.assign(
    has_kw=df['n_keywords'] > 0,
    has_mesh=df['n_mesh'] > 0,
).groupby('year').agg(
    keywords=('has_kw', 'mean'),
    coi=('has_coi', 'mean'),
    mesh=('has_mesh', 'mean'),
) * 100

plt.figure(figsize=(12, 4))
sns.lineplot(data=by_year, dashes=False, markers=False)
plt.title("Field coverage by year (%)"); plt.xlabel("year"); plt.ylabel("% of records")
plt.legend(title=""); plt.tight_layout(); plt.show()

def first_year_at(series, thresh):
    """First year a coverage series reaches `thresh` percent, or None."""
    hit = series.ge(thresh)
    return int(series.index[hit.argmax()]) if hit.any() else None

print(f"keywords first reach 50% coverage: {first_year_at(by_year['keywords'], 50)}")
print(f"COI statements first reach 5% coverage: {first_year_at(by_year['coi'], 5)}")

### 7b. Completeness heatmap
The same story as a year × field grid. "Missing" means the field is **empty** (empty list / False / zero) — these columns are never null in this schema. Darker = more missing.

In [ ]:
present = pd.DataFrame({
    "keywords":     df['n_keywords'] > 0,
    "COI":          df['has_coi'],
    "affiliations": df['affiliations'].map(lambda v: len(v) > 0),
})
missing_by_year = (1 - present.groupby(df['year']).mean()) * 100
plt.figure(figsize=(12, 4))
sns.heatmap(missing_by_year.T, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': '% missing (empty)'})
plt.title('Missing (empty) data by year, sparse fields'); plt.tight_layout(); plt.show()

### 7c. COI statement classification (local corpus only)
A rule-based split of the free-text COI statement. Papers with **no COI field at all** ("no statement") are kept separate from those that **state no conflict** ("states none") — a distinction that matters for any transparency analysis. Needs `coi_statement` text, present only in the local clean corpus; on the metadata-only export the cell skips. The keyword rules are heuristic; treat as indicative, not exact.

In [ ]:
if df['coi_statement'].fillna('').str.len().gt(0).any():
    def classify_coi(text):
        t = (text or "").lower()
        if any(p in t for p in ['no conflict', 'no competing', 'declare no', 'none declared', 'nothing to disclose']):
            return 'states none'
        if any(p in t for p in ['received', 'grants', 'consultant', 'advisory board', 'speaker', 'honoraria', 'stock', 'employment', 'paid']):
            return 'discloses conflict'
        return 'ambiguous'
    # has_coi separates "no field at all" from statements that exist
    df['coi_class'] = np.where(~df['has_coi'], 'no statement',
                               df['coi_statement'].apply(classify_coi))
    print(df['coi_class'].value_counts().to_string())
    coi_ts = df.groupby('year')['coi_class'].value_counts(normalize=True).unstack().fillna(0) * 100
    coi_ts.plot.area(figsize=(12, 4), alpha=0.7)
    plt.title('COI statement classification over time (%)'); plt.xlabel('year'); plt.ylabel('%')
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left'); plt.tight_layout(); plt.show()
else:
    print("coi_statement is empty (metadata-only export) — skipping; run on the local clean corpus.")

## 8. Text fields (titles and abstracts)

### 8a. Abstract length distribution
The original abstract length (`abstract_len`) is retained even in the metadata-only export, so the distribution can always be shown.

In [ ]:
print(f"mean original abstract length: {df['abstract_len'].mean():.0f} chars")
plt.figure(figsize=(10, 3))
sns.histplot(df['abstract_len'].clip(upper=4000), bins=50, color="#8a5fb0")
plt.title("Original abstract length (chars, clipped 4000)"); plt.xlabel("chars")
plt.tight_layout(); plt.show()

### 8b. Title length, and abstract length over time
Titles are always present (the only text in the published release). Abstracts have lengthened steadily over the decades.

In [ ]:
tl = df['title'].fillna('').str.len()
print(f"title length — mean {tl.mean():.0f}, median {int(tl.median())}, "
      f"min {int(tl.min())}, max {int(tl.max())} chars")
plt.figure(figsize=(10, 3))
sns.histplot(tl.clip(upper=300), bins=60, color="#1d6fb8")
plt.title("Title length (chars)"); plt.xlabel("chars"); plt.tight_layout(); plt.show()

abs_year = df.groupby('year')['abstract_len'].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=abs_year.index, y=abs_year.values, marker="o", color="#8a5fb0")
plt.title("Mean abstract length over time (chars)"); plt.xlabel("year"); plt.ylabel("mean chars")
plt.tight_layout(); plt.show()
print(abs_year.round(0).astype(int).to_string())

### 8c. Team size vs abstract length
Tests whether larger teams write longer abstracts. Both quantities are heavy-tailed (a few papers have hundreds of authors or very long abstracts), which would squash a linear plot into one corner; plotting on log–log spreads the dense range so the relationship is visible, and the fit then estimates a power-law rather than additive trend. Expect a weak positive relationship at most.

In [ ]:
s = df[(df['n_authors'] > 0) & (df['abstract_len'] > 0)].sample(min(10000, len(df)), random_state=42)
plt.figure(figsize=(8, 6))
sns.regplot(x=np.log10(s['n_authors']), y=np.log10(s['abstract_len']),
            scatter_kws={'alpha': 0.3, 's': 5}, line_kws={'color': 'red'})
plt.xlabel('log10(authors)'); plt.ylabel('log10(abstract length)')
plt.title('Authors vs abstract length (log–log)'); plt.tight_layout(); plt.show()
print("Pearson r (log-log):",
      round(np.corrcoef(np.log10(s['n_authors']), np.log10(s['abstract_len']))[0, 1], 3))

### 8d. Abstract length by era and team size
Whether abstracts lengthened uniformly or only in certain team-size groups. Median characters by era × author band (uses `abstract_len`, retained in the export, so this runs everywhere).

In [ ]:
band = pd.cut(df['n_authors'], [0, 1, 5, 20, 10**9], labels=["solo", "2–5", "6–20", "21+"])
era = pd.cut(df['year'], [0, 2009, 2019, 2100], labels=["pre-2010", "2010–19", "2020+"])
strat = (df.assign(band=band, era=era)
           .groupby(['era', 'band'], observed=True)['abstract_len'].median().unstack())
print(strat.round(0).astype('Int64').to_string())
strat.T.plot(kind='bar', figsize=(10, 4))
plt.title("Median abstract length (chars) by era and team size")
plt.ylabel("median chars"); plt.xlabel("team size"); plt.legend(title="era")
plt.tight_layout(); plt.show()

### 8e. Abstract word and sentence counts (abstract text required)
Word and sentence counts per abstract, and mean length-in-words over time. Requires abstract text — present locally, skipped on the metadata-only export.

In [ ]:
if HAS_ABSTRACTS:
    wc = df['abstract'].fillna('').str.split().str.len()
    sc = df['abstract'].fillna('').str.count(r'[.!?]')
    print(f"words per abstract — mean {wc.mean():.0f}, median {int(wc.median())}")
    print(f"sentences per abstract — mean {sc.mean():.0f}, median {int(sc.median())}")
    wc_year = df.assign(wc=wc).groupby('year')['wc'].mean()
    plt.figure(figsize=(12, 4))
    sns.lineplot(x=wc_year.index, y=wc_year.values, marker="o", color="#8a5fb0")
    plt.title("Mean abstract length over time (words)"); plt.xlabel("year"); plt.ylabel("mean words")
    plt.tight_layout(); plt.show()
else:
    print("abstract text absent (metadata-only export) — skipping word/sentence analysis.")

### 8f. Structured vs unstructured abstracts (abstract text required)
Many modern abstracts use explicit section labels (Background / Methods / Results / Conclusions). The share that are structured rises over time — useful for any downstream parsing of abstract sections. Label detection is heuristic (presence of the words), so read it as a trend rather than an exact count.

In [ ]:
if HAS_ABSTRACTS:
    labels = ['background', 'methods', 'results', 'conclusion', 'conclusions',
              'objective', 'objectives', 'introduction', 'purpose', 'findings']
    pat = re.compile(r'\b(?:' + '|'.join(labels) + r')\b', re.I)
    df['is_structured'] = df['abstract'].fillna('').str.contains(pat)
    struct_year = df.groupby('year')['is_structured'].mean() * 100
    print(f"overall structured: {df['is_structured'].mean()*100:.0f}%")
    plt.figure(figsize=(12, 4))
    sns.lineplot(x=struct_year.index, y=struct_year.values, marker="o", color="#2a9d5c")
    plt.title("Share of abstracts with section labels, by year (%)")
    plt.xlabel("year"); plt.ylabel("% structured"); plt.tight_layout(); plt.show()
else:
    print("abstract text absent (metadata-only export) — skipping structure detection.")

### 8g. Most common abstract words (abstract text required)
A quick vocabulary view with a small stop-word filter — a rough sense of dominant terms, not a full NLP pipeline (proper tokenization, lemmatization, and stop-word lists belong in the dedicated text notebook). Sampled for speed.

In [ ]:
if HAS_ABSTRACTS:
    STOP = set("the a an and or of to in for with we were was is are be by on at as that this "
               "from been being has have had not but their there which study patients results "
               "methods conclusion conclusions background objective using used also between these "
               "than into more most can may such our both per via".split())
    cnt = collections.Counter()
    for txt in df['abstract'].fillna('').sample(min(50000, len(df)), random_state=0):
        for w in re.findall(r"[a-z]{4,}", txt.lower()):
            if w not in STOP:
                cnt[w] += 1
    topw = pd.Series(dict(cnt.most_common(20)))[::-1]
    plt.figure(figsize=(10, 6))
    sns.barplot(x=topw.values, y=topw.index, color="#8a5fb0")
    plt.title("Most frequent abstract words (sampled, stop-words removed)")
    plt.xlabel("occurrences"); plt.ylabel(""); plt.tight_layout(); plt.show()
else:
    print("abstract text absent (metadata-only export) — skipping vocabulary view.")

### Note — MeSH top-level categories (not included)
A top-level category breakdown (Diseases, Chemicals, Anatomy, …) needs the actual MeSH tree numbers from NLM (the descriptor → tree-number table). The first-letter-of-the-name shortcut does not work — descriptor names do not start with their category letter (e.g. "Neoplasms" is category C, "Humans" is B). Left for a dedicated MeSH-analysis notebook where the tree file can be joined.

## 9. Summary and recommendations

### What this analysis found
- **Volume** grows steadily from the mid-1990s to ~125k US-affiliated articles/year by 2025, excluding the artifact window.
- **The 2013–2015 dip is an indexing artifact** — established journals dropped out of the US-affiliation filter during PubMed's ~2013–2014 affiliation-indexing change, not a real decline. Treat pre-2014 volumes as lower bounds.
- **Team science dominates by 2025** — solo authorship fell 14.6%→2.5%; the 6–20 author band is now the majority.
- **Abstracts lengthened ~45%** (≈1,134→1,639 chars, 1994→2025).
- **MeSH depth was stable ~13 descriptors through 2019, then fell to ~8** (NLM automated indexing, not a content change).
- **Journals are broadly distributed** — top 100 ≈ 19% of output across 7,316 journals (Gini in 4e).
- **Keyword and COI fields are sparse before ~2012 and ~2017** respectively (auto-detected in 7a).

### What the corpus is good for
Bibliometrics, MeSH-based topic exploration, author/affiliation network analysis, and NLP pipelines (re-fetch abstracts by PMID). Best avoided for: precise year-over-year volume claims spanning 2013–2015, and month-level timing on imprecise-date records.

### Downstream caveats
- **2013–2015 volume** — exclude or flag for any longitudinal rate analysis.
- **~77% of records have imprecise dates** — use `pubdate_precision`; restrict to `full_date` for month-level work.
- **No abstract text in the published release** — text methods need the local build or a re-fetch by PMID.
- **Author names are not disambiguated** — "J Smith" is not unified across records.
- **Keywords** valid from ~2012; early records contain non-biomedical noise (filter before use).
- **MeSH** depth changes after 2019 — segment by era; drop generic descriptors ("Humans", "Female", …) before clustering.
- **COVID-19** MeSH exists only from 2020 — union with "Coronavirus Infections", "SARS Virus", "Betacoronavirus" for the pre-2020 baseline.
- **Co-authorship networks** — restrict to post-2014; large consortia (max ~2,929 authors) create hub nodes.